# 📈 Market Price Forecasting Model Training

In [ ]:
import pandas as pd
import numpy as np
import pickle
import os
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.preprocessing import MinMaxScaler
print('Libraries loaded!')

In [ ]:
df = pd.read_csv('../datasets/market_prices.csv')
crop_df = df[df['crop_name'] == 'Paddy (Dhan)(Common)'].copy()
crop_df['date'] = pd.to_datetime(crop_df['date'])
df_ts = crop_df.groupby('date')['price'].mean().reset_index().sort_values('date')
print(f'Time-series data points: {len(df_ts)}')

In [ ]:
prices = df_ts['price'].values.reshape(-1, 1)
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_prices = scaler.fit_transform(prices)

LOOKBACK = 30
X, y = [], []
for i in range(LOOKBACK, len(scaled_prices)):
    X.append(scaled_prices[i-LOOKBACK:i, 0])
    y.append(scaled_prices[i, 0])
X = np.array(X)
y = np.array(y)
X = X.reshape(X.shape[0], X.shape[1], 1)

# Sequential split to prevent future information leakage
split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]
print(f'X_train shape: {X_train.shape}, X_test shape: {X_test.shape}')

In [ ]:
model = models.Sequential([
    layers.Input(shape=(LOOKBACK, 1)),
    layers.LSTM(32, return_sequences=False),
    layers.Dropout(0.2),
    layers.Dense(1)
])
model.compile(optimizer='adam', loss='mse')

early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
model.fit(X_train, y_train, epochs=30, batch_size=32, validation_data=(X_test, y_test), callbacks=[early_stopping], verbose=1)
print('LSTM trained with early stopping!')

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

pred_train_scaled = model.predict(X_train)
pred_test_scaled = model.predict(X_test)

pred_train = scaler.inverse_transform(pred_train_scaled).flatten()
actual_train = scaler.inverse_transform(y_train.reshape(-1, 1)).flatten()
pred_test = scaler.inverse_transform(pred_test_scaled).flatten()
actual_test = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()

plt.figure(figsize=(14, 6))
plt.plot(range(len(actual_test)), actual_test, label='Actual Price (Test)', color='teal', lw=2)
plt.plot(range(len(pred_test)), pred_test, label='Predicted Price (Test)', color='crimson', linestyle='--', lw=2)
plt.title('Paddy Market Price Forecasting - Test Set (Telangana Average)', fontsize=14)
plt.xlabel('Timeline (Days)')
plt.ylabel('Price (INR / Quintal)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('price_forecast_plot.png', dpi=150)
plt.show()

print("="*60)
print("LSTM TIME SERIES EVALUATION METRICS")
print("="*60)
print(f"Train R² score:                {r2_score(actual_train, pred_train):.4f}")
print(f"Test R² score:                 {r2_score(actual_test, pred_test):.4f}")
print(f"Test Mean Absolute Error:      {mean_absolute_error(actual_test, pred_test):.2f} INR/quintal")
print(f"Test Root Mean Squared Error:  {np.sqrt(mean_squared_error(actual_test, pred_test)):.2f} INR/quintal")

In [ ]:
os.makedirs('../../ml_models', exist_ok=True)
model.save('../../ml_models/price_prediction.h5')
with open('../../ml_models/price_prediction_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print('LSTM model and scaler saved successfully!')